<div style="padding:20px; border-radius:6px; background-color:#ffe6e6; color:#b30000; border-left:5px solid #b30000;">
<b>Jalankan notebook ini dengan environment <code>mandala_env</code>.</b><br/>
Siapkan file <code>.env</code> di <b>root workdir proyek Anda</b> berisi <code>USEROPENDAP</code> dan <code>PASSOPENDAP</code>.
</div>

# Tutorial: Akses Data INACAWO via OPeNDAP

Materi hands-on ini menyambung sesi *Metode Akses Data INACAWO* (PPT pelatihan OFS).
Di sini Anda belajar membaca produk forecast INACAWO dari server Hyrax BMKG Maritim,
memahami struktur datanya, lalu membuat peta singkat bergaya operasional OFS.

**Urutan belajar**
1. OPeNDAP vs Mandala (kapan pilih yang mana)
2. Struktur data Atmos / Wave / Flows
3. Autentikasi & buka dataset remote
4. Hands-on tiga produk + inspect atribut
5. Case study ringan: briefing metocean Selat Sunda
6. Troubleshooting


## 1. OPeNDAP vs Mandala — pilih pintu yang tepat

| | **OPeNDAP (Hyrax)** | **Mandala (S3 + Zarr)** |
|---|---|---|
| Bentuk | File NetCDF per cycle, dipisah Atmos / Wave / Flows | Living archive `zcollection` (00/12 ditumpuk) |
| Retensi | Pendek (~1–3 bulan forecast operasional) | Panjang (arsip + hindcast) |
| Akses tipikal | Laptop / tools DAP / subset jarak jauh | Mandalahub, I/O paralel di jaringan lokal |
| Dimensi | `time`, `lat`, `lon` (+ `depth` pada Flows) | `TSTEP`, `LAY`, `ROW`, `COL` |
| Nama variabel | Pendek (`hs`, `uwnd`, `u`) | Panjang (`sig_wave_height`, `ubar_eastward`, …) |

**Best practice (ingat pre-test):**
- OPeNDAP → **subset di server**, jangan unduh file multi-GB utuh ke workspace.
- Mandala → baca Zarr langsung via S3 di Mandalahub, jangan unduh OPeNDAP lalu upload ulang.

Katalog: [https://maritim.bmkg.go.id/opendap/inaCawo/contents.html](https://maritim.bmkg.go.id/opendap/inaCawo/contents.html)


## 2. Struktur data INACAWO di OPeNDAP

Domain display INACAWO ≈ **3 km**, grid horizontal tipikal **2201 × 1201** (lon × lat),
cakupan sekitar **90–145°E / 15°S–15°N**.

### Produk file
| Produk | Pola nama | Dimensi | Isi utama |
|---|---|---|---|
| Atmos | `InaCAWOAtmos_YYYYMMDD_HHMM.nc` | `time, lat, lon` | angin 10 m, gust, MSLP, suhu, RH, hujan |
| Wave | `InaCAWOWave_*_3hourly.nc` (sering) | `time, lat, lon` | `hs`, `hmax`, periode, arah, partisi swell |
| Flows | `InaCAWOFlows_*.nc` | `time, depth, lat, lon` | arus `u`/`v`, `T`, `S`, `zeta` |

Cycle operasional umum: **00 / 06 / 12 / 18 UTC**. Folder katalog biasanya `inaCawo/YYYY/MM/`.

### Satuan — hati-hati, tidak selalu seragam
- **Atmos** `uwnd`/`vwnd`/`gust`: **m/s** (`M/S`)
- **Wave** `uwnd`/`vwnd`: sering **knots** (beda dari Atmos!)
- **Wave** `hs` / `hmax`: **meter**
- **Flows** `u`/`v`: **cm/s**; `T`: Celsius; `zeta`: m

Plot operasional OFS menampilkan angin dalam **knot** (konversi dari m/s ≈ `× 1.94384`).

### Konvensi arah
- Komponen `uwnd`/`vwnd` dan `u`/`v`: CF **eastward / northward** (vektor *ke arah*).
- Field `dir` gelombang di **Hyrax OPeNDAP**: attrs sering kosong; nilai empiris = sudut matematika
  (**CCW dari timur**, sudah arah jalar / *going to*). Beda dari Mandala `*_dir` yang CF
  **meteorological coming from** (CW dari utara).
- Relasi kasar: `meteo_from ≈ (270 − opendap_dir) % 360`.
- Saat plot: OPeNDAP pakai `wave_dir_convention="math_to"`; Mandala pakai default `"meteo_from"`.

### Mapping singkat ke Mandala (preview)
| Konsep | OPeNDAP | Mandala (contoh) |
|---|---|---|
| Significant wave height | `hs` | `sig_wave_height` |
| Wave direction | `dir` | `*_dir` |
| Wind U/V | `uwnd` / `vwnd` | `U_WIND` / `V_WIND` (WRF) |
| Current U/V | `u` / `v` | `u_sur_eastward` / `v_sur_northward` |
| SSH | `zeta` | `zeta` |


## 3. Persiapan: import & kredensial

Kredensial dibaca dari `.env` di **root workdir proyek Anda** (`USEROPENDAP`, `PASSOPENDAP`).
Path home user tidak di-hardcode — helper `load_project_env()` mencari folder yang berisi `training_ofs/`.

**Jangan** print password ke output notebook. Jangan share `.env` antar peserta.


In [ ]:
from datetime import datetime, timedelta
from pathlib import Path
from urllib.parse import quote

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import xarray as xr

import os
from training_ofs import (
    MPS_TO_KNOT,
    find_project_root,
    get_param,
    list_params,
    load_project_env,
    plot_ofs_map,
)

# Portable: resolve this workdir's project root (no hardcoded /home/<user>/...)
PROJECT_ROOT = find_project_root()
ENV_PATH = load_project_env(
    required_keys=("USEROPENDAP", "PASSOPENDAP"),
)
USER = os.environ["USEROPENDAP"]
PASS = os.environ["PASSOPENDAP"]

print("Project root:", PROJECT_ROOT)
print("Loaded env:", ENV_PATH if ENV_PATH.is_file() else "(env vars only)")
print("OPeNDAP user:", USER)
print("Parameter OFS tersedia:", ", ".join(list_params()))


### Helper URL OPeNDAP

Hyrax di BMKG memakai Basic Auth. Pola yang andal di `xarray`/`netCDF4`:
sisipkan user/password di URL (di-quote agar karakter khusus aman).


In [ ]:
OPENDAP_ROOT = "maritim.bmkg.go.id/opendap/inaCawo"


def opendap_url(relpath: str) -> str:
    """Build authenticated OPeNDAP URL for a path under /inaCawo/."""
    relpath = relpath.lstrip("/")
    return (
        f"https://{quote(USER, safe='')}:{quote(PASS, safe='')}"
        f"@{OPENDAP_ROOT}/{relpath}"
    )


def open_inacawo(relpath: str, **kwargs) -> xr.Dataset:
    """Open a remote INACAWO NetCDF via OPeNDAP."""
    return xr.open_dataset(opendap_url(relpath), decode_times=False, **kwargs)


def summarize_attrs(ds: xr.Dataset) -> pd.DataFrame:
    rows = []
    for name, da in ds.data_vars.items():
        rows.append(
            {
                "variable": name,
                "dims": tuple(da.dims),
                "shape": tuple(da.shape),
                "units": da.attrs.get("units"),
                "standard_name": da.attrs.get("standard_name"),
                "long_name": da.attrs.get("long_name"),
            }
        )
    return pd.DataFrame(rows)


# Example cycle used throughout this notebook (adjust if missing on server)
YEAR, MONTH, CYCLE = 2026, 9, "20260912_0000"
print("Contoh cycle:", CYCLE)


## 4. Hands-on Atmos

Buka satu file Atmos, inspect dimensi/atribut, lalu subset kecil di sekitar Selat Sunda.


In [ ]:
atmos_path = f"{YEAR}/{MONTH:02d}/InaCAWOAtmos_{CYCLE}.nc"
ds_atmos = open_inacawo(atmos_path)
print(ds_atmos)
summarize_attrs(ds_atmos)


Perhatikan: angin Atmos bersatuan **m/s**. Untuk plot OFS (`ws`) kita set `wind_in_mps=True`.


In [ ]:
# Subset ringan: 1 timestep + bbox Selat Sunda (~3x3 deg)
t0 = 12  # ~12 jam setelah inisialisasi (sesuaikan)
lat_slice = slice(-7.5, -5.0)
lon_slice = slice(104.5, 107.5)

atm = ds_atmos.isel(time=t0).sel(lat=lat_slice, lon=lon_slice)
print(atm[["uwnd", "vwnd", "gust", "mslp"]])
print("uwnd units:", atm["uwnd"].attrs.get("units"))


In [ ]:
# Quicklook sederhana (bukan layout OFS) — pastikan subset sudah kecil
atm["uwnd"].plot(figsize=(6, 4))
plt.title("Atmos uwnd (m/s) — quicklook subset")
plt.show()


## 5. Hands-on Wave

File Wave sering memakai sufiks `_3hourly`. Cek satuan angin di file Wave — bisa **knots**,
berbeda dari Atmos.


In [ ]:
wave_path = f"{YEAR}/{MONTH:02d}/InaCAWOWave_{CYCLE}_3hourly.nc"
ds_wave = open_inacawo(wave_path)
print(ds_wave)
summarize_attrs(ds_wave)


In [ ]:
# time index: pilih mendekati t+12 jam jika memungkinkan
wave_t = min(4, ds_wave.sizes["time"] - 1)
wav = ds_wave.isel(time=wave_t).sel(lat=lat_slice, lon=lon_slice)
print("hs units:", wav["hs"].attrs.get("units"))
print("wave uwnd units:", wav["uwnd"].attrs.get("units"))
wav["hs"].plot(figsize=(6, 4))
plt.title("Wave hs (m) — quicklook subset")
plt.show()


## 6. Hands-on Flows

Flows punya dimensi **depth**. Untuk case pelayaran singkat, ambil level permukaan (index 0)
dan komponen arus `u`/`v` (biasanya **cm/s** di OPeNDAP).


In [ ]:
flows_path = f"{YEAR}/{MONTH:02d}/InaCAWOFlows_{CYCLE}.nc"
ds_flows = open_inacawo(flows_path)
print(ds_flows)
print("depth values:", ds_flows["depth"].values)
summarize_attrs(ds_flows)


In [ ]:
flo = (
    ds_flows.isel(time=t0, depth=0)
    .sel(lat=lat_slice, lon=lon_slice)
)
print("u units:", flo["u"].attrs.get("units"))
speed = np.sqrt(flo["u"] ** 2 + flo["v"] ** 2)
speed.plot(figsize=(6, 4))
plt.title("Flows |u,v| permukaan (cm/s) — quicklook")
plt.show()


## 7. Case study: briefing metocean jalur Selat Sunda

**Skenario.** Forecaster diminta ringkasan singkat kondisi angin, gelombang, dan arus
di jalur Selat Sunda untuk horizon dekat (satu waktu forecast).

**Aturan beban:** bbox kecil, 1 waktu, 1–2 variabel per produk — lalu plot dengan
**layout operasional OFS** (logo, title, colorbar, legend arah).


In [ ]:
# Waktu metadata untuk header peta (approx dari nama cycle + index)
baserun = datetime.strptime(CYCLE, "%Y%m%d_%H%M")
# Atmos/Flows hourly → t0 jam; Wave 3-hourly → wave_t * 3
fcst_atmos = baserun + timedelta(hours=int(t0))
fcst_wave = baserun + timedelta(hours=int(wave_t) * 3)

lon_a = atm["lon"].values
lat_a = atm["lat"].values

fig_ws, ax_ws = plot_ofs_map(
    lon_a,
    lat_a,
    np.sqrt(atm["uwnd"].values ** 2 + atm["vwnd"].values ** 2),
    param="ws",
    u=atm["uwnd"].values,
    v=atm["vwnd"].values,
    wind_in_mps=True,  # Atmos OPeNDAP = m/s
    baserun=baserun,
    forecast=fcst_atmos,
    wind_input="INACAWO",
    map_area="Selat Sunda",
    quiver_skip=4,
)
plt.show()


In [ ]:
fig_hs, ax_hs = plot_ofs_map(
    wav["lon"].values,
    wav["lat"].values,
    wav["hs"].values,
    param="swh",
    direction=wav["dir"].values if "dir" in wav else None,
    # Hyrax `dir` = math CCW-from-East (going to), bukan meteo-from seperti Mandala
    wave_dir_convention="math_to",
    baserun=baserun,
    forecast=fcst_wave,
    wind_input="INACAWO",
    map_area="Selat Sunda",
    quiver_skip=4,
)
plt.show()


In [ ]:
# Flows OPeNDAP u/v sudah cm/s → current_in_mps=False
fig_csd, ax_csd = plot_ofs_map(
    flo["lon"].values,
    flo["lat"].values,
    np.sqrt(flo["u"].values ** 2 + flo["v"].values ** 2),
    param="csd",
    u=flo["u"].values,
    v=flo["v"].values,
    current_in_mps=False,
    baserun=baserun,
    forecast=fcst_atmos,
    wind_input="INACAWO",
    map_area="Selat Sunda",
    quiver_skip=4,
)
plt.show()


### Interpretasi singkat (isi sendiri saat praktik)

Tulis 3–5 kalimat berdasarkan ketiga peta di atas, misalnya:
- Kekuatan & arah angin dominan di jalur
- Tinggi gelombang signifikan (`hs`) relatif terhadap ambang operasional lokal
- Arus permukaan — apakah mendukung / menentang alur pelayaran

**Catatan arah:** panah = arah jalar gelombang (*going to*). OPeNDAP Hyrax `dir` diplot dengan
`wave_dir_convention="math_to"`; Mandala `*_dir` dengan `"meteo_from"` — nilai derajat beda,
tapi panah harus selaras di lokasi yang sama.


## 8. Checklist & troubleshooting

| Gejala | Kemungkinan | Perbaikan |
|---|---|---|
| HTTP 401 | User/pass salah / belum di `.env` | Cek `USEROPENDAP`/`PASSOPENDAP` |
| File not found | Cycle belum ada / nama Wave tanpa `_3hourly` | Browse katalog Hyrax, sesuaikan path |
| Sangat lambat / RAM penuh | Subset terlalu lebar / load full grid | Crop lat/lon + 1 waktu **sebelum** plot |
| Colorbar angin “aneh” | Salah satuan (m/s vs knot) | Atmos: `wind_in_mps=True`; Wave wind: cek attrs |
| Arus terlalu kecil/besar | cm/s vs m/s | Cek `units`; set `current_in_mps` sesuai |

**Lanjut:** buka `mandala-tutorial.ipynb` untuk akses living archive via S3/Zarr dan bandingkan naming/dimensi dengan OPeNDAP.


In [ ]:
# Tutup handle remote (baik untuk sesi panjang)
ds_atmos.close()
ds_wave.close()
ds_flows.close()
print("Datasets closed.")
